# Data Mining Project 9

## Project Overview
This notebook presents the complete implementation of the uploaded Data Mining Project 9. The original project code and analysis are retained while the notebook is reorganized into clear sections for dataset understanding, preprocessing, exploratory analysis, model development, evaluation, and conclusion.


## 1. Import Required Libraries

The libraries used in the original project are loaded first so that data processing, visualization, and machine learning operations can be performed in an organized workflow.


## 2. Load and Inspect the Dataset


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

## 3. Data Cleaning and Preprocessing


In [ ]:
path = kagglehub.dataset_download(
    "mkechinov/ecommerce-events-history-in-electronics-store")

print("Dataset path:", path)
print("Files:", os.listdir(path))

## 4. Exploratory Data Analysis


In [ ]:
# Find the CSV file automatically
csv_files = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))
df.head()

## 5. Feature Engineering / Data Preparation


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

## 6. Model Development


In [ ]:
df.info()

## 7. Model Evaluation


In [ ]:
df.describe(include="all").T

## 8. Results and Analysis


In [ ]:
# Convert event_time to datetime
df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")

# Remove duplicate records
df = df.drop_duplicates()

# Remove rows without the essential recommendation fields
df = df.dropna(subset=["user_id", "product_id", "event_type"])

print("Shape after preprocessing:", df.shape)
print("\nMissing values:")
print(df[["user_id", "product_id", "event_type"]].isnull().sum())

## 9. Final Output / Prediction


In [ ]:
event_counts = df["event_type"].value_counts()

print(event_counts)

plt.figure(figsize=(7, 4))
event_counts.plot(kind="bar")
plt.title("Customer Event Distribution")
plt.xlabel("Event Type")
plt.ylabel("Number of Events")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Most popular products based on customer interactions
top_products = (
    df["product_id"]
    .value_counts()
    .head(10)
)

print("Top 10 Most Interacted Products:")
print(top_products)

In [ ]:
# Most popular brands
if "brand" in df.columns:
    top_brands = df["brand"].dropna().value_counts().head(10)
    print("Top 10 Brands:")
    print(top_brands)

    plt.figure(figsize=(8, 4))
    top_brands.plot(kind="bar")
    plt.title("Top 10 Brands by Customer Interactions")
    plt.xlabel("Brand")
    plt.ylabel("Interactions")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
event_weights = {
    "view": 1,
    "cart": 3,
    "purchase": 5,
    "remove_from_cart": 0.5
}

df["interaction_score"] = df["event_type"].map(event_weights).fillna(0)

interaction_df = (
    df.groupby(["user_id", "product_id"])["interaction_score"]
    .sum()
    .reset_index()
)

interaction_df.head()

In [ ]:
# Keep the most frequently interacted products to make the
# recommendation matrix efficient while retaining popular products.
top_product_ids = (
    interaction_df.groupby("product_id")["interaction_score"]
    .sum()
    .sort_values(ascending=False)
    .head(3000)
    .index
)

model_df = interaction_df[
    interaction_df["product_id"].isin(top_product_ids)
].copy()

# Keep users with at least two product interactions
user_counts = model_df.groupby("user_id")["product_id"].nunique()
active_users = user_counts[user_counts >= 2].index

model_df = model_df[model_df["user_id"].isin(active_users)].copy()

print("Modeling rows:", len(model_df))
print("Unique users:", model_df["user_id"].nunique())
print("Unique products:", model_df["product_id"].nunique())

In [ ]:
user_index = {
    user_id: index
    for index, user_id in enumerate(model_df["user_id"].unique())
}

product_index = {
    product_id: index
    for index, product_id in enumerate(model_df["product_id"].unique())
}

rows = model_df["user_id"].map(user_index)
cols = model_df["product_id"].map(product_index)
values = model_df["interaction_score"]

user_product_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_index), len(product_index))
)

print("Interaction matrix shape:", user_product_matrix.shape)
print("Non-zero interactions:", user_product_matrix.nnz)

In [ ]:
# Transpose so each row represents a product
product_user_matrix = user_product_matrix.T.tocsr()

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

model.fit(product_user_matrix)

print("Recommendation model trained successfully.")

In [ ]:
reverse_product_index = {
    index: product_id
    for product_id, index in product_index.items()
}

def recommend_products(user_id, n_recommendations=5):
    if user_id not in user_index:
        return pd.DataFrame(columns=["product_id", "score"])

    user_row = user_product_matrix[user_index[user_id]]
    interacted_indices = user_row.indices

    if len(interacted_indices) == 0:
        return pd.DataFrame(columns=["product_id", "score"])

    scores = {}

    for product_idx in interacted_indices:
        distances, neighbors = model.kneighbors(
            product_user_matrix[product_idx],
            n_neighbors=11
        )

        for distance, neighbor_idx in zip(distances[0], neighbors[0]):
            neighbor_idx = int(neighbor_idx)

            if neighbor_idx == product_idx:
                continue

            similarity = 1 - float(distance)
            product_id = reverse_product_index[neighbor_idx]

            if neighbor_idx not in interacted_indices:
                scores[product_id] = scores.get(product_id, 0) + similarity

    recommendations = (
        pd.DataFrame(
            list(scores.items()),
            columns=["product_id", "score"]
        )
        .sort_values("score", ascending=False)
        .head(n_recommendations)
        .reset_index(drop=True)
    )

    return recommendations

In [ ]:
# Select a customer who has enough interaction history
sample_user = model_df["user_id"].value_counts().index[0]

print("Selected customer:", sample_user)

print("\nPreviously interacted products:")
print(
    model_df[model_df["user_id"] == sample_user]
    .sort_values("interaction_score", ascending=False)
    [["product_id", "interaction_score"]]
    .head(10)
)

In [ ]:
recommendations = recommend_products(sample_user, 5)

print("Top 5 Recommended Products:")
recommendations

In [ ]:
# Add available product information such as category, brand and price.
product_details = (
    df[["product_id", "category_code", "brand", "price"]]
    .drop_duplicates("product_id")
)

final_recommendations = recommendations.merge(
    product_details,
    on="product_id",
    how="left"
)

final_recommendations

In [ ]:
def precision_at_5(user_id):
    user_data = (
        model_df[model_df["user_id"] == user_id]
        .sort_values("interaction_score", ascending=False)
    )

    if len(user_data) < 3:
        return None

    hidden_product = user_data.iloc[-1]["product_id"]
    history_products = set(user_data.iloc[:-1]["product_id"])

    # Temporarily use the customer's history directly
    # to calculate candidate recommendations.
    scores = {}

    for product_id in history_products:
        product_idx = product_index[product_id]

        distances, neighbors = model.kneighbors(
            product_user_matrix[product_idx],
            n_neighbors=11
        )

        for distance, neighbor_idx in zip(distances[0], neighbors[0]):
            neighbor_idx = int(neighbor_idx)

            if reverse_product_index[neighbor_idx] in history_products:
                continue

            similarity = 1 - float(distance)
            candidate = reverse_product_index[neighbor_idx]
            scores[candidate] = scores.get(candidate, 0) + similarity

    top5 = sorted(scores, key=scores.get, reverse=True)[:5]

    return int(hidden_product in top5)

In [ ]:
eligible_users = (
    model_df.groupby("user_id")["product_id"]
    .nunique()
)

eligible_users = eligible_users[eligible_users >= 3].index[:100]

results = [
    precision_at_5(user_id)
    for user_id in eligible_users
]

results = [value for value in results if value is not None]

precision_5 = np.mean(results) if results else 0

print("Evaluated users:", len(results))
print("Precision@5:", round(precision_5, 4))

# Conclusion

The Data Mining Project 9 notebook was reorganized into a structured end-to-end workflow. The original implementation is retained, with clearer project sections covering dataset handling, preprocessing, analysis, model development, and evaluation. This format makes the notebook easier to execute, understand, present, and explain during a project demonstration or viva.

The final results should be interpreted from the outputs generated after running the notebook from top to bottom. No unsupported performance values have been added.
